## 1. Dataset

In [1]:
import pandas as pd
from itertools import product

ages         = ['young', 'pre-presbyopic', 'presbyopic']
prescriptions = ['myope', 'hypermetrope']
astigmatisms  = ['no', 'yes']
tear_rates    = ['reduced', 'normal']

# Labels na ordem canônica (product: age > prescription > astigmatism > tear_rate)
# Verificadas contra as regras do PDF de solução (hard=4, soft=5, none=15)
labels = [
    'none', 'soft', 'none', 'hard',   # young,          myope
    'none', 'soft', 'none', 'hard',   # young,          hypermetrope
    'none', 'soft', 'none', 'hard',   # pre-presbyopic, myope
    'none', 'soft', 'none', 'none',   # pre-presbyopic, hypermetrope
    'none', 'none', 'none', 'hard',   # presbyopic,     myope
    'none', 'soft', 'none', 'none',   # presbyopic,     hypermetrope
]

records = []
for i, (age, pres, astig, tear) in enumerate(
        product(ages, prescriptions, astigmatisms, tear_rates)):
    records.append({
        'age': age,
        'spectacle_prescription': pres,
        'astigmatism': astig,
        'tear_production_rate': tear,
        'lenses': labels[i]
    })

df = pd.DataFrame(records)
print(f"Total de exemplos: {len(df)}")
print(df['lenses'].value_counts().rename('contagem'))
df


Total de exemplos: 24
lenses
none    15
soft     5
hard     4
Name: contagem, dtype: int64


,age,spectacle_prescription,astigmatism,tear_production_rate,lenses
0,young,myope,no,reduced,none
1,young,myope,no,normal,soft
2,young,myope,yes,reduced,none
3,young,myope,yes,normal,hard
4,young,hypermetrope,no,reduced,none
5,young,hypermetrope,no,normal,soft
6,young,hypermetrope,yes,reduced,none
7,young,hypermetrope,yes,normal,hard
8,pre-presbyopic,myope,no,reduced,none
9,pre-presbyopic,myope,no,normal,soft


In [2]:
def prism_induce_rules(df, target_col, target_class):
    """Induz regras PRISM para uma classe-alvo. Retorna lista de regras (lista de tuplas)."""
    attributes = [c for c in df.columns if c != target_col]
    working_set = df.copy()
    rules = []

    while (working_set[target_col] == target_class).sum() > 0:
        antecedent = []
        current_subset = working_set.copy()
        remaining_attrs = list(attributes)

        while True:
            best_prob, best_coverage, best_pair = -1, -1, None
            for attr in remaining_attrs:
                for val in current_subset[attr].unique():
                    subset_f = current_subset[current_subset[attr] == val]
                    n_class = (subset_f[target_col] == target_class).sum()
                    n_total = len(subset_f)
                    if n_total == 0:
                        continue
                    prob = n_class / n_total
                    if prob > best_prob or (prob == best_prob and n_class > best_coverage):
                        best_prob, best_coverage, best_pair = prob, n_class, (attr, val)

            if best_pair is None:
                break

            attr, val = best_pair
            antecedent.append((attr, val))
            current_subset = current_subset[current_subset[attr] == val].copy()
            remaining_attrs = [a for a in remaining_attrs if a != attr]

            prob_final = (current_subset[target_col] == target_class).sum() / len(current_subset)
            if prob_final == 1.0 or len(remaining_attrs) == 0:
                break

        rules.append(antecedent)

        # Remove exemplos cobertos pela regra
        mask = pd.Series([True] * len(working_set), index=working_set.index)
        for attr, val in antecedent:
            mask &= (working_set[attr] == val)
        working_set = working_set[~mask].copy()

    return rules


def format_rule(antecedent, target_col, target_class):
    cond = ' AND '.join(f'{a} = {v}' for a, v in antecedent)
    return f'IF {cond} THEN {target_col} = {target_class}'


print("Funções prism_induce_rules() e format_rule() definidas.")


Funções prism_induce_rules() e format_rule() definidas.


## 3. Geração das Regras para Cada Classe

In [3]:
all_rules = {}

for cls in ['hard', 'soft', 'none']:
    rules = prism_induce_rules(df, 'lenses', cls)
    all_rules[cls] = rules
    print(f"{'='*60}")
    print(f"  Classe alvo: lenses = {cls}")
    print(f"{'='*60}")
    for i, rule in enumerate(rules, 1):
        print(f"  R{i}: {format_rule(rule, 'lenses', cls)}")
    print()

print("\n✅ Total de regras induzidas:", sum(len(v) for v in all_rules.values()))


  Classe alvo: lenses = hard
  R1: IF astigmatism = yes AND tear_production_rate = normal AND spectacle_prescription = myope THEN lenses = hard
  R2: IF age = young AND astigmatism = yes AND tear_production_rate = normal THEN lenses = hard

  Classe alvo: lenses = soft
  R1: IF astigmatism = no AND tear_production_rate = normal AND spectacle_prescription = hypermetrope THEN lenses = soft
  R2: IF astigmatism = no AND tear_production_rate = normal AND age = young THEN lenses = soft
  R3: IF age = pre-presbyopic AND astigmatism = no AND tear_production_rate = normal THEN lenses = soft

  Classe alvo: lenses = none
  R1: IF tear_production_rate = reduced THEN lenses = none
  R2: IF age = presbyopic AND tear_production_rate = normal AND spectacle_prescription = myope AND astigmatism = no THEN lenses = none
  R3: IF spectacle_prescription = hypermetrope AND astigmatism = yes AND age = pre-presbyopic THEN lenses = none
  R4: IF age = presbyopic AND spectacle_prescription = hypermetrope AND a

## 4. Análise Passo a Passo — Classe `hard`

Reprodução detalhada das tabelas de `p(hard | atributo = valor)` a cada iteração,
conforme o documento de solução do PDF.


In [4]:
def prism_verbose(df, target_col, target_class):
    """Versão detalhada do PRISM com saída explicativa por passo."""
    attributes = [c for c in df.columns if c != target_col]
    working_set = df.copy()
    rule_num = 0

    while (working_set[target_col] == target_class).sum() > 0:
        rule_num += 1
        print(f"\n{'━'*65}")
        print(f"  Regra {rule_num}  |  Subconjunto atual: {len(working_set)} exemplos "
              f"({(working_set[target_col]==target_class).sum()} positivos)")
        print(f"{'━'*65}")

        antecedent = []
        current_subset = working_set.copy()
        remaining_attrs = list(attributes)
        step = 0

        while True:
            step += 1
            rows = []
            for attr in remaining_attrs:
                for val in current_subset[attr].unique():
                    filt = current_subset[current_subset[attr] == val]
                    n_pos = (filt[target_col] == target_class).sum()
                    n_tot = len(filt)
                    if n_tot == 0: continue
                    prob = n_pos / n_tot
                    rows.append({
                        'atributo = valor': f'{attr} = {val}',
                        f'p({target_class}|attr=val)': f'{n_pos}/{n_tot} = {prob:.2f}',
                        'cobertura_pos': n_pos
                    })
            tbl = (pd.DataFrame(rows)
                   .sort_values(['cobertura_pos'], ascending=False)
                   .reset_index(drop=True))
            print(f"\n  Passo {step}  |  Antecedente: {antecedent or '(vazio)'}")
            print(tbl.to_string(index=False))

            # Escolha
            best_prob, best_coverage, best_pair = -1, -1, None
            for attr in remaining_attrs:
                for val in current_subset[attr].unique():
                    filt = current_subset[current_subset[attr] == val]
                    n_pos = (filt[target_col] == target_class).sum()
                    n_tot = len(filt)
                    if n_tot == 0: continue
                    prob = n_pos / n_tot
                    if prob > best_prob or (prob == best_prob and n_pos > best_coverage):
                        best_prob, best_coverage, best_pair = prob, n_pos, (attr, val)

            if best_pair is None: break
            attr, val = best_pair
            print(f"\n  ✔ Escolhido: {attr} = {val}  ({best_coverage}/{len(current_subset[current_subset[attr]==val])} = {best_prob:.2f})")

            antecedent.append((attr, val))
            current_subset = current_subset[current_subset[attr] == val].copy()
            remaining_attrs = [a for a in remaining_attrs if a != attr]

            p_final = (current_subset[target_col] == target_class).sum() / len(current_subset)
            if p_final == 1.0 or len(remaining_attrs) == 0:
                break

        coverage_pos = (current_subset[target_col] == target_class).sum()
        print(f"\n  🎯 REGRA: {format_rule(antecedent, target_col, target_class)}")
        print(f"     Cobertura: {coverage_pos} exemplo(s) positivos")

        mask = pd.Series([True]*len(working_set), index=working_set.index)
        for attr, val in antecedent:
            mask &= (working_set[attr] == val)
        working_set = working_set[~mask].copy()


prism_verbose(df, 'lenses', 'hard')



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Regra 1  |  Subconjunto atual: 24 exemplos (4 positivos)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Passo 1  |  Antecedente: (vazio)
                     atributo = valor p(hard|attr=val)  cobertura_pos
        tear_production_rate = normal      4/12 = 0.33              4
                    astigmatism = yes      4/12 = 0.33              4
       spectacle_prescription = myope      3/12 = 0.25              3
                          age = young       2/8 = 0.25              2
                 age = pre-presbyopic       1/8 = 0.12              1
spectacle_prescription = hypermetrope      1/12 = 0.08              1
                     age = presbyopic       1/8 = 0.12              1
                     astigmatism = no      0/12 = 0.00              0
       tear_production_rate = reduced      0/12 = 0.00              0

  ✔ Escolhido: astigmatism = yes  (4/12 = 0.33)

  Passo 2  |  Antecede

## 5. Resumo Final das Regras Induzidas

In [5]:
header = f"{'Classe':<8}  {'Nº':<4}  Regra"
print(header)
print("─" * 80)
for cls in ['hard', 'soft', 'none']:
    for i, rule in enumerate(all_rules[cls], 1):
        print(f"  {cls:<8}  R{i:<3}  {format_rule(rule, 'lenses', cls)}")
    print()


Classe    Nº    Regra
────────────────────────────────────────────────────────────────────────────────
  hard      R1    IF astigmatism = yes AND tear_production_rate = normal AND spectacle_prescription = myope THEN lenses = hard
  hard      R2    IF age = young AND astigmatism = yes AND tear_production_rate = normal THEN lenses = hard

  soft      R1    IF astigmatism = no AND tear_production_rate = normal AND spectacle_prescription = hypermetrope THEN lenses = soft
  soft      R2    IF astigmatism = no AND tear_production_rate = normal AND age = young THEN lenses = soft
  soft      R3    IF age = pre-presbyopic AND astigmatism = no AND tear_production_rate = normal THEN lenses = soft

  none      R1    IF tear_production_rate = reduced THEN lenses = none
  none      R2    IF age = presbyopic AND tear_production_rate = normal AND spectacle_prescription = myope AND astigmatism = no THEN lenses = none
  none      R3    IF spectacle_prescription = hypermetrope AND astigmatism = yes AND a

## 6. Predição e Avaliação

In [6]:
def predict_prism(instance, rules_by_class, default='none'):
    """Classifica uma instância usando as regras PRISM (ordem: hard > soft > none)."""
    for cls in ['hard', 'soft', 'none']:
        for rule in rules_by_class[cls]:
            if all(instance.get(attr) == val for attr, val in rule):
                return cls
    return default


df['predicted'] = df.apply(lambda row: predict_prism(row.to_dict(), all_rules), axis=1)
correct = (df['predicted'] == df['lenses']).sum()
print(f"Acurácia no conjunto de treinamento: {correct}/{len(df)} = {correct/len(df):.1%}")
print()

# Matriz de confusão
from collections import Counter
classes = ['hard', 'soft', 'none']
print(f"{'Real \\ Predito':<20}" + "".join(f"{c:>10}" for c in classes))
print("─" * 50)
for real in classes:
    counts = Counter(df[df['lenses'] == real]['predicted'])
    print(f"  {real:<18}" + "".join(f"{counts.get(p, 0):>10}" for p in classes))


Acurácia no conjunto de treinamento: 24/24 = 100.0%

Real \ Predito            hard      soft      none
──────────────────────────────────────────────────
  hard                       4         0         0
  soft                       0         5         0
  none                       0         0        15


In [7]:
# Teste com instância nova
nova = {
    'age': 'young',
    'spectacle_prescription': 'myope',
    'astigmatism': 'yes',
    'tear_production_rate': 'normal',
}

pred = predict_prism(nova, all_rules)
print("Instância de teste:")
for k, v in nova.items():
    print(f"  {k} = {v}")
print(f"\nClassificação PRISM → lenses = {pred}")


Instância de teste:
  age = young
  spectacle_prescription = myope
  astigmatism = yes
  tear_production_rate = normal

Classificação PRISM → lenses = hard
